Test Place Recognition and Hierarchical Localization on the ITLP-Campus dataset using `opr.pipelines`

In [1]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-03-15 21:31:51.788 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


# Outdoor

## MinkLoc3D

### Prepare databases

#### Weights download

You can download the `minkloc3d_nclt.pth` from the HuggingFace model hub:
https://huggingface.co/OPR-Project/PlaceRecognition-NCLT.

```bash
wget https://huggingface.co/OPR-Project/PlaceRecognition-NCLT/resolve/main/minkloc3d_nclt.pth
```

#### Dataset download

You can download the dataset:

- Kaggle:
  - [ITLP Campus Outdoor](https://www.kaggle.com/datasets/alexandermelekhin/itlp-campus-outdoor)
- Hugging Face:
  - [ITLP Campus Outdoor](https://huggingface.co/datasets/OPR-Project/ITLP-Campus-Outdoor)


In [2]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
index_path = "/home/kartashov_ga/projects/tests/gsloc/14-03-26/"

In [3]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [4]:
# dataloaders = {}

# for track in TRACK_LIST:
#     dataset = ITLPCampus(
#         dataset_root=f"{dataset_path}/{track}",
#         sensors=["front_cam"],
#         mink_quantization_size=0.5,
#         max_point_distance=40.0,
#         image_transform=image_transform_fn,
#         load_semantics=False,
#         load_text_descriptions=False,
#         load_text_labels=False,
#         load_aruco_labels=False,
#         indoor=True,
#     )
#     dataloaders[track] = DataLoader(
#         dataset, batch_size=16, shuffle=False, num_workers=4, collate_fn=dataset.collate_fn
#     )


In [5]:
three_rscan_ds = ThreeRScan(
    dataset_root="/mnt/external_usb_hdd/6YL/Datasets/3RScan",
    meta_path = index_path,
    rebuild_meta=False,  # meta.parquet already built
    limit=20000,
    image_transform=image_transform_fn
)
# dataloader = DataLoader(
#     three_rscan_ds, batch_size=16, shuffle=False, num_workers=4, collate_fn=three_rscan_ds.collate_fn
# )

In [6]:
model = MegaLoc()
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [7]:
# descriptors = []
# for image_path in tqdm(cuted_df['path']):
#     image = read_image(image_path)
#     batch = to_batch(image)
#     batch = {k: v.to(device) for k, v in batch.items()}
#     with torch.no_grad():
#         desc = model(batch)
#     descriptors.append(desc["final_descriptor"].cpu().numpy())
# descriptors = np.concatenate(descriptors, axis=0)
# np.save(root_data_dir / "descriptors.npy", descriptors)

# print(f"Descriptors saved to {root_data_dir / 'descriptors.npy'}")

In [7]:
index = FaissFlatIndex.generate(
    directory=index_path,
    dataset=three_rscan_ds,
    model=model,
    rebuild_meta=False)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-03-15 21:32:15.211 | INFO     | mmpr.inference.index:generate:391 - Using existing meta.parquet
2026-03-15 21:32:15.212 | INFO     | mmpr.inference.index:generate:418 - Using existing descriptors.npy
2026-03-15 21:32:15.331 | INFO     | mmpr.inference.index:generate:436 - schema.json file was saved in /home/kartashov_ga/projects/tests/gsloc/14-03-26/


Index created at /home/kartashov_ga/projects/tests/gsloc/14-03-26/
Index size: 20000, dim: 8448 metric: l2


In [8]:
pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
)

In [9]:
three_rscan_q = ThreeRScan(
    dataset_root="/mnt/external_usb_hdd/6YL/Datasets/3RScan",
    # meta_path = index_path,
    rebuild_meta=True,
    limit=10000,
    image_transform=image_transform_fn
)

2026-03-15 21:33:03.692 | INFO     | gsloc.datasets.three_rscan:__init__:147 - Rebuilding metadata for 3rscan dataset
2026-03-15 21:33:03.692 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:90 - Scanning 3rscan dataset...
2026-03-15 21:33:06.656 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:118 - Scanned 10000 rows


In [10]:
out = pipeline.infer(three_rscan_q[9000])

In [11]:
out

PlaceRecognitionResult(descriptor=array([ 0.00143673,  0.00783881,  0.02143787, ...,  0.01978837,
       -0.00375643, -0.0045516 ], shape=(8448,), dtype=float32), indices=array([9000, 8787, 8786, 9001, 9005]), distances=array([8.8346681e-09, 5.0083792e-01, 7.0701224e-01, 7.5864244e-01,
       8.1422341e-01], dtype=float32), db_idx=array([9000, 8787, 8786, 9001, 9005]), db_pose=array([[ 0.798643  ,  0.983432  , -0.132033  ,  0.75250036, -0.5780175 ,
        -0.25381622, -0.18766014],
       [ 0.356377  ,  1.48875   , -0.131014  ,  0.8410546 , -0.427336  ,
        -0.3099954 , -0.11795727],
       [ 0.382755  ,  1.40634   , -0.0807452 ,  0.8403163 , -0.41481936,
        -0.32412416, -0.12937145],
       [ 0.841705  ,  0.995071  , -0.140846  ,  0.7504945 , -0.58146584,
        -0.24821363, -0.19247201],
       [ 0.860654  ,  0.989988  , -0.125784  ,  0.7427527 , -0.56332934,
        -0.28782725, -0.21939473]], dtype=float32))

In [8]:
descriptors = []
with torch.no_grad():
    for batch in tqdm(dataloader):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        final_descriptor = model(batch)["final_descriptor"]
        descriptors.append(final_descriptor.detach().cpu().numpy())
descriptors = np.concatenate(descriptors, axis=0)
N, D = descriptors.shape

Path(index_path).mkdir(parents=True, exist_ok=True)
np.save(f"{index_path}/descriptors.npy", descriptors)

100%|██████████| 1250/1250 [06:42<00:00,  3.11it/s]


In [9]:
# Minimal schema.json
schema = {"version": "1", "dim": D, "metric": "l2", "created_at": "26/02/2026", "opr_version": ""}
Path(f"{index_path}/schema.json").write_text(json.dumps(schema))

# Load the index
index = FaissFlatIndex.load(index_path)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")


Index created at /home/kartashov_ga/projects/tests/gsloc/13-03-26/
Index size: 20000, dim: 8448 metric: l2


# Test PlaceRecognitionPipeline

In [16]:
from typing import Tuple
import numpy as np
from scipy.spatial.transform import Rotation

def pose_to_matrix(pose):
    position = pose[:3]
    orientation_quat = pose[3:]
    rotation = Rotation.from_quat(orientation_quat)
    pose_matrix = np.eye(4)
    pose_matrix[:3,:3] = rotation.as_matrix()
    pose_matrix[:3,3] = position
    return pose_matrix


def compute_error(estimated_pose, gt_pose):
    estimated_pose = pose_to_matrix(estimated_pose)
    gt_pose = pose_to_matrix(gt_pose)
    error_pose = np.linalg.inv(estimated_pose) @ gt_pose
    dist_error = np.sum(error_pose[:3, 3]**2) ** 0.5
    r = Rotation.from_matrix(error_pose[:3, :3])
    rotvec = r.as_rotvec()
    angle_error = (np.sum(rotvec**2)**0.5) * 180 / np.pi
    angle_error = abs(90 - abs(angle_error-90))
    return dist_error, angle_error


In [17]:
ij_permutations = list(itertools.permutations(range(len(TRACK_LIST)), 2))

median_dist_errors = []
median_angle_errors = []
mean_dist_errors = []
mean_angle_errors = []

#for i, j in tqdm(ij_permutations[:1], position=0):
local_dist_errors = []
local_angle_errors = []
database = TRACK_LIST[0]
query = "01_2023-11-09-twilight"
index = FaissFlatIndex.load(f"/home/kartashov_ga/projects/tests/gsloc/12-03-26/{database}")

pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
)

query_dataset = ITLPCampus(
    dataset_root=f"/mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor/{query}",
    sensors=["front_cam"],
    mink_quantization_size=0.5,
    max_point_distance=40.0,
    load_semantics=False,
    load_text_descriptions=False,
    load_text_labels=False,
    load_aruco_labels=False,
    indoor=True,
)

for sample in tqdm(query_dataset, position=1):
    print(sample)
    out = pipeline.infer(sample)
    dist_error, angle_error = compute_error(out["pose"], sample["pose"].numpy())
    local_dist_errors.append(dist_error)
    local_angle_errors.append(angle_error)

median_dist_errors.append(np.median(local_dist_errors))
median_angle_errors.append(np.median(local_angle_errors))
mean_dist_errors.append(np.mean(local_dist_errors))
mean_angle_errors.append(np.mean(local_angle_errors))


  0%|          | 0/1310 [00:00<?, ?it/s]

{'idx': tensor(0), 'pose': tensor([ 0.3394,  0.0110,  0.0345,  0.0036,  0.0019, -0.0017,  1.0000]), 'image_front_cam': tensor([[[ 0.2111,  0.2453,  0.2624,  ...,  0.1939,  0.2111,  0.2111],
         [ 0.1939,  0.1768,  0.2111,  ...,  0.1768,  0.1768,  0.1939],
         [ 0.2453,  0.2453,  0.2453,  ...,  0.1768,  0.2111,  0.2453],
         ...,
         [-0.4397, -0.4739, -0.4397,  ..., -0.0629, -0.0629, -0.0629],
         [-0.4739, -0.5253, -0.4739,  ..., -0.0287, -0.0287,  0.0056],
         [-0.4568, -0.4568, -0.5253,  ...,  0.0227, -0.0287, -0.0116]],

        [[ 0.3803,  0.4153,  0.4328,  ...,  0.3627,  0.3803,  0.3803],
         [ 0.3627,  0.3452,  0.3803,  ...,  0.3452,  0.3452,  0.3627],
         [ 0.4153,  0.4153,  0.4153,  ...,  0.3452,  0.3803,  0.4153],
         ...,
         [-0.2675, -0.3025, -0.2675,  ...,  0.1001,  0.1001,  0.1001],
         [-0.2850, -0.3375, -0.3025,  ...,  0.1176,  0.1352,  0.1527],
         [-0.2675, -0.2675, -0.3375,  ...,  0.1527,  0.1176,  0.1352]]

TypeError: 'PlaceRecognitionResult' object is not subscriptable

In [ ]:
median_dist_errors, mean_dist_errors


([2.7146281571568993], [16.442354230465188])

In [ ]:
median_angle_errors, mean_angle_errors


([5.768895097717426], [12.568010270903558])